# Cuaderno 4 — Interpolación de Newton (Integrante B)
**Workshop 2 · Algoritmos Numéricos · Ingeniería Civil · Universidad del Quindío**
**Ejercicios enlazados:** Ej. 12 (3 nodos), 13 (4 nodos), 14 (incorporación de un nodo: casos A y B). **Módulos:** `interpolacion.py`, `datos.py`, `metricas.py`. Ejecutar con *Kernel → Restart & Run All*.

## 1. Teoría
$f[x_i]=y_i,\quad f[x_i,\dots,x_{i+k}]=\dfrac{f[x_{i+1},\dots,x_{i+k}]-f[x_i,\dots,x_{i+k-1}]}{x_{i+k}-x_i}$, y
$P_n(x)=\sum_{k=0}^{n}f[x_0,\dots,x_k]\prod_{i<k}(x-x_i)$ (evaluación anidada/Horner). Al **añadir un nodo al final** se conservan todos los coeficientes anteriores y se añade uno.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from numpy.polynomial import polynomial as P
from scipy.interpolate import BarycentricInterpolator, lagrange as sp_lagrange, CubicSpline
from interpolacion import *
from datos import cargar, control_calidad, particion_comun, comparar, fraccion_regular, codigos_calificacion
import metricas
pd.options.display.float_format = '{:.5f}'.format

In [ ]:
def mostrar(x, y, titulo):
    T = diferencias_divididas(x, y); m = len(x)
    t = pd.DataFrame(T, columns=["f[ ]"] + [f"orden {j}" for j in range(1, m)]); t.insert(0, "x_i", x); print(titulo); display(t.fillna(""))
    return T[0]
c12 = mostrar([1,2,3], [1,4,9], "Ej. 12 (3 nodos)")
print("Manual: 1, 3, 1 -> P=x^2 | código:", c12)
c13 = mostrar([0,1,2,4], [-1,1,5,31], "Ej. 13 (4 nodos)")
print("Manual: -1, 2, 1, 1/2 -> P=x^3/2 - x^2/2 + 2x - 1 | código:", c13)
x13 = np.array([0,1,2,4.]); print("P(3) código:", evaluar_newton(x13, c13, [3.])[0], " manual:", 27/2 - 9/2 + 6 - 1)

## 2. Ejercicio 14 — Incorporación de un nodo (casos A y B)

In [ ]:
for nom, (xn, yn) in {"A: nodo (5,33)": (5, 33), "B: nodo (4,16)": (4, 16)}.items():
    x2, y2, c2, ok = agregar_nodo([1,2,3], [1,4,9], xn, yn)
    print(nom, "| coeficientes:", np.round(c2, 5), "| ¿se conservan los 3 anteriores?", ok)
print("Manual A: orden 3 = 1/3 | Manual B: orden 3 = 0")

**Qué se conserva:** $f[x_0]=1,\,f[x_0,x_1]=3,\,f[x_0,x_1,x_2]=1$ en ambos casos; solo se calcula la nueva diagonal. En el caso B el nodo cae sobre $y=x^2$ y el coeficiente nuevo es 0; en el A no, y aparece $\tfrac13(x-1)(x-2)(x-3)$.

## 3. Contraste con biblioteca y reproducción de nodos

In [ ]:
for nom, x, y in [("Ej. 12", [1,2,3], [1,4,9]), ("Ej. 13", [0,1,2,4], [-1,1,5,31]), ("14A", [1,2,3,5], [1,4,9,33])]:
    x = np.array(x, float); y = np.array(y, float); c = coef_newton(x, y); xx = np.linspace(x.min(), x.max(), 60)
    print(nom, "| max|Newton - Barycentric| =", np.abs(evaluar_newton(x, c, xx) - BarycentricInterpolator(x, y)(xx)).max(),
          "| residual en nodos =", np.abs(evaluar_newton(x, c, x) - y).max())

## 4. Aplicación a datos reales (mismos nodos que Lagrange y spline)

In [ ]:
df, es_real = cargar(n_recorte=36); nodos, val = particion_comun(len(df), 4, 2); R = comparar(df, nodos, val)
tn, yn = R["t"][nodos], R["y"][nodos]
print("Ventana de 5 nodos consecutivos (tabla de diferencias):"); mostrar(tn[:5], yn[:5], "Newton local")
print(f"Newton global con {len(nodos)} nodos: residual en nodos = {R['reproduccion_nodos']['Newton']:.2e} ft³/s; |Lagrange−Newton|máx = {R['acuerdo']['|Lagrange−Newton|máx']:.2e}")
pd.options.display.float_format = '{:.3f}'.format; display(R["metricas"])

## 5. Cierre
**Comprobé:** tablas manuales = tablas del código, conservación de coeficientes, acuerdo con SciPy y reproducción de nodos. **Límites:** los ejercicios 12–14 usan datos casi polinómicos, por lo que no muestran las dificultades del polinomio global con muchos nodos; esas aparecen con los datos reales (Cuaderno 5). Newton y Lagrange son el mismo polinomio: su diferencia mide la implementación, no el error frente al fenómeno. El residual en nodos de Newton global es pequeño pero no nulo por el mal condicionamiento.

**Registro de IA:** ver `registro_IA.md` y `history.md`. *(Completar con fecha, herramienta/modelo tal como se muestra, prompt exacto, qué se aceptó/modificó/descartó y la comprobación independiente; o declarar «Sin uso de IA».)*